In [26]:
import pandas as pd 
import numpy as np
import yfinance as yf
import statsmodels.api as sm

# Baixando dados SP500 do ano em questão
year = 2019
sp500_dr = yf.download(tickers="^GSPC", start=f"{str(year)}-01-01", end=f"{str(year)}-12-31")["Close"].pct_change()[1:]
sp500_dr.columns = ["rm"]

# Função que calcula o beta
def beta_ols(rp, rm):
    df = rp.to_frame("rp").join(rm, how="inner")
    y = df["rp"]
    X = sm.add_constant(df["rm"])
    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 5}
    )
    return model.params["rm"]

# Pegando retornos pros portfólios
returns = pd.read_parquet(f"../../data/02_clean/returns_{str(year)}.parquet")
modes = ["fast", "medium", "slow"]
centralities = ["central", "peripheral"]
returns_dict = {}

for centrality in centralities:
    for mode in modes:
        tickers = pd.read_csv(f"../../data/07_portfolios_metadata/{centrality}_{mode}_metadata.csv")["Ticker"]
        returns_dict[f"{centrality} {mode}"] = returns[tickers]

[*********************100%***********************]  1 of 1 completed


In [27]:
beta_dict = {}

for portfolio_name, df_returns in returns_dict.items():
    betas = {}

    for ticker in df_returns.columns:
        rp = df_returns[ticker].dropna()

        if rp.empty:
            betas[ticker] = np.nan
            continue

        try:
            betas[ticker] = beta_ols(rp, sp500_dr)
        except Exception:
            betas[ticker] = np.nan

    # transforma dict em DataFrame
    beta_dict[portfolio_name] = (
        pd.DataFrame.from_dict(betas, orient="index", columns=[f"beta_{year}"])
    )

In [28]:
beta_df = (
    pd.concat(beta_dict, names=["portfolio", "Ticker"])
    .reset_index()
)
beta_df.to_csv("../../data/07_portfolios_metadata/beta_df.csv", index=False)